In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"

FIGSIZE = (20,18)
DPI = 100
GENERATE_PLOTS = False

In [ ]:
import pandas as pd
import geopandas as gpd
import sys
import json
from shapely.geometry import shape
from hotelling.spatial.admin import join_lor_names

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')
PATH_PROCESSED = REPO_ROOT / Path('data/processed')

zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_PROCESSED / 'lor.parquet')

with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')

grid = gpd.read_parquet(PATH_PROCESSED / 'pop_grid.parquet')


In [ ]:
from hotelling.spatial.census import build_grid_polygons

grid = build_grid_polygons(grid)
grid['index'] = grid.index


In [ ]:
from hotelling.spatial.osm import fetch_pois, process_supermarkets, CHAIN_TYPE_MAP

# Fetch raw supermarkets (cached to data/raw/OSM_POIs_Berlin_supermarket.parquet)
pois_raw = fetch_pois(type="supermarket", city="Berlin")
print(f"Raw supermarkets from OSM: {len(pois_raw)}")

# Normalise, clip to grid, classify by chain type
supermarkets = process_supermarkets(pois_raw, grid)
print(f"Supermarkets in grid with recognised chain: {len(supermarkets)}")
print(supermarkets['chain'].value_counts().to_string())

supermarkets.to_parquet(PATH_PROCESSED / 'supermarkets.parquet', index=False)
print("Saved: supermarkets.parquet")


In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

if GENERATE_PLOTS:
    # Plot these supermarkets
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    supermarkets.plot(ax=ax, color='firebrick', label='Supermarkets', alpha=0.7, markersize=1)
    grid.plot(ax=ax, facecolor='royalblue', edgecolor='none', label='Grid', alpha=0.2)
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', label='Berlin Boundary', linewidth=2)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)

    ax.set_axis_off()
    ax.legend()
    plt.title("Supermarkets in Berlin with Grid Overlay")
    plt.show()


In [ ]:
if GENERATE_PLOTS:
    # Plot the supermarkets with normalized chain names
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    supermarkets.plot(ax=ax, column='chain', categorical=True, legend=True, markersize=5, alpha=0.7)
    grid.plot(ax=ax, facecolor='royalblue', edgecolor='none', label='Grid', alpha=0.2)
    berlin.plot(ax=ax, facecolor='none', edgecolor='black', label='Berlin Boundary', linewidth=2)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha = 0.3)
    ax.set_axis_off()
    ax.legend()
    plt.title("Supermarkets in Berlin with Normalized Chain Names")
    plt.show()


In [ ]:
from hotelling.spatial.assembly import add_lcc_layer
from hotelling.spatial.osm import fetch_pois

lcc_gdf = fetch_pois(type="LCC", city="Berlin")
print(f"LCC POIs: {len(lcc_gdf)} (all types)")
print(f"  shop=mall: {(lcc_gdf['shop'] == 'mall').sum()}")

grid_malls = add_lcc_layer(grid, lcc_gdf)
print(f"Cells with mall overlap: {grid_malls['has_mall'].sum()}")
grid_malls[grid_malls['has_mall']].head()


In [ ]:
import matplotlib.pyplot as plt
import contextily as ctx

grid_only_malls = grid_malls[grid_malls['has_mall']]
mall_overlay = lcc_gdf[lcc_gdf['shop'] == 'mall'].to_crs(grid.crs)

if GENERATE_PLOTS:
    fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
    grid_malls.to_crs(berlin.crs).plot(ax=ax, color='royalblue', legend=True, alpha=0.2)
    #mall_overlay.to_crs(berlin.crs).plot(ax=ax, color='firebrick', label='Malls', alpha=1)
    grid_only_malls.to_crs(berlin.crs).plot(ax=ax, column='mall_area', cmap='OrRd', markersize=5, legend=True, alpha=0.7)
    berlin.plot(ax=ax, color='none', edgecolor='black', linewidth=1.5)
    ctx.add_basemap(ax, crs=berlin.crs, source=ctx.providers.OpenStreetMap.Mapnik, zoom=11, zorder=0, alpha=0.5)
    ax.set_axis_off()
    plt.show()


In [ ]:
if not GENERATE_PLOTS:
    import nbformat, pathlib

    _nb_path = pathlib.Path(__file__) if "__file__" in dir() else None
    # Fallback: set explicitly if auto-detection unavailable
    _nb_path = pathlib.Path("GEO_03_OSM.ipynb")  # ← set once per notebook

    _nb = nbformat.read(_nb_path, as_version=4)
    for _cell in _nb.cells:
        _cell["outputs"] = []
        _cell["execution_count"] = None
    nbformat.write(_nb, _nb_path)
    print(f"Outputs cleared: {_nb_path.name}")